# MongoDB sampling estimates

Start the toolkit Docker MongoDB and initialize its `demo.business` fixture (see the local databases guide). This notebook reads that collection and temporarily creates a sample in `dbrtoolkit_samples`. It removes that sample afterward. The synchronous interface works directly in Jupyter. Small samples here illustrate the API, not accurate estimates.

In [ ]:
from dbrtoolkit.mongodb import (
    MongoCollectionRef,
    MongoExecutor,
    MongoFindQuery,
    MongoSampleEstimator,
)

target = MongoCollectionRef("demo", "business")
query = MongoFindQuery(target, {"stars": {"$gte": 4}}, limit=1)
executor = MongoExecutor.from_uri("mongodb://localhost:27017")

## Full population versus a fresh sample
The find limit is deliberately one: estimation measures filter matches **before** limit. Omitting sampling options gives full-population evaluation.

In [ ]:
with MongoSampleEstimator(executor, target) as estimator:
    exact = estimator.estimate(query)

with MongoSampleEstimator(executor, target, sample_ratio=0.5) as estimator:
    fresh = estimator.estimate(query)

[(name, r.cardinality, r.selectivity, r.sample_matches, r.sample_size)
 for name, r in [("full", exact), ("sample", fresh)]]

## Reuse one sample
A persisted sample avoids drawing new documents for every query. The context manager prepares it and cleans it up, even if an estimate fails. Estimates from a tiny sample can differ substantially from full-population counts.

In [ ]:
with MongoSampleEstimator(
    executor, target, sample_size=4, persist=True,
) as estimator:
    sample_target = estimator.sample_target
    first = estimator.estimate(query)
    second = estimator.estimate(query)
    print("Temporary collection:", sample_target)
    print(first)
    assert first == second

print("Sample removed:", estimator.sample_target is None)

## Explicit preparation and cleanup
You can also manage the lifetime yourself. Use one sample control only; oversized samples and ratios that round to zero issue warnings.

In [ ]:
estimator = MongoSampleEstimator(executor, target, sample_ratio=0.5)
try:
    estimator.prepare()
    print(estimator.estimate(query))
finally:
    estimator.close()
    executor.close()

## Scripts and cleanup
The same synchronous code works in notebooks and scripts. The estimator shares its executor’s background loop. Close the estimator before the executor; context managers automate cleanup, including on interruption.

The population count is recorded at preparation; keep source data stable. Zero sample matches are not proof of zero population matches.
